In [1]:
import polars as pl
from pathlib import Path

# 1. Definir rutas relativas
BRONZE_DIR = Path("../data/01_bronze")
DICT_PATH = Path("../data/Diccionario_gastos.csv") # Archivo dado por la misma Pagina

# Configuración visual
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(10)

# 2. Leer el diccionario y generar el esquema automáticamente
try:
    df_dict = pl.read_csv(DICT_PATH)
    
    # Filtramos las variables que son de tipo "Carácter" según el MEF
    columnas_texto = (
        df_dict.filter(pl.col("TIPO_DATO").str.strip_chars() == "Carácter")
        .select("VARIABLE")
        .to_series()
        .to_list()
    )
    
    codigos_a_texto = {col: pl.String for col in columnas_texto}
    print(f"✅ Diccionario cargado. Se forzarán {len(codigos_a_texto)} columnas a formato Texto.")

except Exception as e:
    print(f"Error al leer el diccionario: {e}")
    codigos_a_texto = {}

# 3. Escanear Bronze con el esquema dinámico
try:
    lazy_mef = pl.scan_csv(
        BRONZE_DIR / "*.csv",
        separator=",",
        infer_schema_length=10000,
        encoding="utf8",
        ignore_errors=True,
        low_memory=False,
        schema_overrides=codigos_a_texto
    )
    print("Grafo de ejecución creado exitosamente con el esquema del MEF.")
except Exception as e:
    print(f"Error al escanear los datos: {e}")

✅ Diccionario cargado. Se forzarán 42 columnas a formato Texto.
Grafo de ejecución creado exitosamente con el esquema del MEF.


In [7]:
# Inspeccionar que el diccionario hizo su magia
print("--- Esquema inferido por Polars (Primeras 15 variables) ---")
schema = lazy_mef.collect_schema()
for i, (col_name, dtype) in enumerate(list(schema.items())[:90]):
    print(f"{i+1}. {col_name}: {dtype}")

# Vista previa de 5 filas usando los 16 hilos de tu procesador
print("\n--- Vista previa de los datos ---")
df_preview = lazy_mef.head(30).collect()
display(df_preview)

--- Esquema inferido por Polars (Primeras 15 variables) ---
1. KEY_VALUE: String
2. NIVEL_GOBIERNO: String
3. NIVEL_GOBIERNO_NOMBRE: String
4. SECTOR: String
5. SECTOR_NOMBRE: String
6. PLIEGO: String
7. PLIEGO_NOMBRE: String
8. EJECUTORA: String
9. EJECUTORA_NOMBRE: String
10. SEC_EJEC: String
11. DEPARTAMENTO_EJECUTORA: String
12. DEPARTAMENTO_EJECUTORA_NOMBRE: String
13. PROVINCIA_EJECUTORA: String
14. PROVINCIA_EJECUTORA_NOMBRE: String
15. DISTRITO_EJECUTORA: String
16. DISTRITO_EJECUTORA_NOMBRE: String
17. PROGRAMA_PPTO: Int64
18. PROGRAMA_PPTO_NOMBRE: String
19. TIPO_ACT_PROY: Int64
20. TIPO_ACT_PROY_NOMBRE: String
21. PRODUCTO_PROYECTO: Int64
22. PRODUCTO_PROYECTO_NOMBRE: String
23. ACTIVIDAD_ACCION_OBRA: Int64
24. ACTIVIDAD_ACCION_OBRA_NOMBRE: String
25. FUNCION: String
26. FUNCION_NOMBRE: String
27. DIVISION_FUNCIONAL: String
28. DIVISION_FUNCIONAL_NOMBRE: String
29. GRUPO_FUNCIONAL: String
30. GRUPO_FUNCIONAL_NOMBRE: String
31. META: String
32. META_NOMBRE: String
33. DEPARTA

KEY_VALUE,NIVEL_GOBIERNO,NIVEL_GOBIERNO_NOMBRE,SECTOR,SECTOR_NOMBRE,PLIEGO,PLIEGO_NOMBRE,EJECUTORA,EJECUTORA_NOMBRE,SEC_EJEC,…,COMPROMETIDO_2025,DEVENGADO_2025,GIRADO_2025,PIA_2026,PIM_2026,CERTIFICADO_2026,COMPROMETIDO_ANUAL_2026,COMPROMETIDO_2026,DEVENGADO_2026,GIRADO_2026
str,str,str,str,str,str,str,str,str,str,…,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64
"""47F40D7C3F80780D9A7DE6E39FBA38…","""E""","""GOBIERNO NACIONAL""","""09""","""ECONOMIA Y FINANZAS""","""059""","""ORGANISMO SUPERVISOR DE LAS CO…","""001""","""ORGANISMO SUPERVISOR DE LAS CO…","""1275""",…,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
"""0D4BFBEB8647B1911AB9CCCCBEB4F2…","""E""","""GOBIERNO NACIONAL""","""10""","""EDUCACION""","""518""","""U.N. AGRARIA LA MOLINA""","""001""","""UNIVERSIDAD NACIONAL AGRARIA L…","""96""",…,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
"""C772437E728E2BFC62019AA6D591D1…","""E""","""GOBIERNO NACIONAL""","""13""","""AGRARIO Y DE RIEGO""","""160""","""SERVICIO NACIONAL DE SANIDAD A…","""001""","""SERVICIO NACIONAL DE SANIDAD A…","""157""",…,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
"""30FC6A30B51F677BC77592D6F6F5E8…","""E""","""GOBIERNO NACIONAL""","""36""","""TRANSPORTES Y COMUNICACIONES""","""036""","""MINISTERIO DE TRANSPORTES Y CO…","""007""","""MTC- PRO VIAS NACIONAL""","""1078""",…,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
"""39D2C461E5D52E69702465A25EE74D…","""E""","""GOBIERNO NACIONAL""","""22""","""MINISTERIO PUBLICO""","""022""","""MINISTERIO PUBLICO""","""002""","""MINISTERIO PUBLICO-GERENCIA GE…","""200""",…,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""B9B92899FA04905B15BA84C0724943…","""E""","""GOBIERNO NACIONAL""","""39""","""MUJER Y POBLACIONES VULNERABLE…","""039""","""MINISTERIO DE LA MUJER Y POBLA…","""001""","""MINISTERIO DE LA MUJER Y POBLA…","""1087""",…,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
"""DA7F5E118085D52E371F06EB671ADD…","""E""","""GOBIERNO NACIONAL""","""10""","""EDUCACION""","""546""","""U.N. DE JAEN""","""001""","""UNIVERSIDAD NACIONAL DE JAEN""","""1364""",…,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
"""7B36955C0765E74C81D7FDDD01ABB6…","""E""","""GOBIERNO NACIONAL""","""32""","""OFICINA NACIONAL DE PROCESOS E…","""032""","""OFICINA NACIONAL DE PROCESOS E…","""001""","""OFICINA NACIONAL DE PROCESOS E…","""479""",…,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0


In [3]:
# --- AUDITORÍA DE GRANULARIDAD (RAW DATA) ---

# 1. Contamos el total de filas reales vs el total de KEY_VALUE distintos
auditoria = lazy_mef.select([
    pl.len().alias("total_filas"),
    pl.col("KEY_VALUE").n_unique().alias("keys_unicas")
]).collect()

total = auditoria["total_filas"][0]
unicas = auditoria["keys_unicas"][0]

print(f"Total de registros: {total:,}")
print(f"Total de KEY_VALUE únicos: {unicas:,}")

if total == unicas:
    print("Excelente: El KEY_VALUE es perfectamente único en el formato ancho.")
else:
    duplicados = total - unicas
    print(f"ALERTA: Hay {duplicados:,} filas con KEY_VALUE duplicado en los datos crudos.")

Total de registros: 8,002,563
Total de KEY_VALUE únicos: 7,842,582
ALERTA: Hay 159,981 filas con KEY_VALUE duplicado en los datos crudos.


In [4]:
# --- AUDITORÍA PROFUNDA SOBRE DATOS LIMPIOS ---

# 1. Aseguramos que la limpieza maestra esté aplicada y pasamos TODAS las columnas a minúsculas
columnas_raw = lazy_mef.collect_schema().names()
lazy_limpio = lazy_mef.rename({col: col.lower() for col in columnas_raw}).with_columns(
    cs.string()
    .str.strip_chars()
    .str.to_lowercase()
    .str.replace_all("á", "a").str.replace_all("é", "e")
    .str.replace_all("í", "i").str.replace_all("ó", "o")
    .str.replace_all("ú", "u")
)

print("⏳ Buscando un duplicado en el dataset YA LIMPIO...")
# 2. Buscamos un duplicado en el dataset limpio
conteo_limpio = (
    lazy_limpio
    .group_by("key_value")
    .agg(pl.len().alias("repeticiones"))
    .filter(pl.col("repeticiones") > 1)
    .head(1) # Sacamos solo 1 para investigar
).collect(engine="streaming")

key_rebelde_limpio = conteo_limpio["key_value"][0]

# 3. Extraemos las filas de ese duplicado
df_muestra_limpia = (
    lazy_limpio
    .filter(pl.col("key_value") == key_rebelde_limpio)
).collect(engine="streaming")

# 4. Buscamos dónde están las diferencias reales
diferencias_limpias = []
for col in df_muestra_limpia.columns:
    if df_muestra_limpia[col].n_unique() > 1:
        diferencias_limpias.append(col)

print(f"🚨 Columnas con diferencias DESPUÉS de limpiar todo el texto a minúsculas y sin tildes:\n{diferencias_limpias}")

# Mostramos la tabla final para que la evalúes
display(df_muestra_limpia.select(["key_value"] + diferencias_limpias))

NameError: name 'cs' is not defined

In [ ]:
# --- DETECTOR REAL DE GRANULARIDAD ---

# 1. Volvemos a procesar el dataset limpio en minúsculas
columnas_raw = lazy_mef.collect_schema().names()
lazy_limpio = lazy_mef.rename({col: col.lower() for col in columnas_raw}).with_columns(
    cs.string()
    .str.strip_chars()
    .str.to_lowercase()
    .str.replace_all("á", "a").str.replace_all("é", "e")
    .str.replace_all("í", "i").str.replace_all("ó", "o")
    .str.replace_all("ú", "u")
)

print("⏳ Pasando escáner para capturar un duplicado real activo...")
# 2. Conseguimos el KEY_VALUE duplicado que SÍ existe ahorita en memoria
conteo_real = (
    lazy_limpio
    .group_by("key_value")
    .agg(pl.len().alias("repeticiones"))
    .filter(pl.col("repeticiones") > 1)
    .head(1)
).collect(engine="streaming")

hash_existente = conteo_real["key_value"][0]

# 3. Extraemos sus filas
df_diagnostico = lazy_limpio.filter(pl.col("key_value") == hash_existente).collect(engine="streaming")

# 4. Filtramos dinámicamente para ver SOLO las columnas de dinero que tengan datos (> 0)
columnas_visibles = ["key_value"]
for col in df_diagnostico.columns:
    if any(col.endswith(f"_{anio}") for anio in ["2022", "2023", "2024", "2025", "2026"]):
        if df_diagnostico[col].cast(pl.Float64).sum() > 0:
            columnas_visibles.append(col)

print(f"🎯 ¡Clon encontrado! Analizando la granularidad del KEY_VALUE: '{hash_existente}'")
display(df_diagnostico.select(columnas_visibles))

⏳ Pasando escáner para capturar un duplicado real activo...
🎯 ¡Clon encontrado! Analizando la granularidad del KEY_VALUE: '2a7fc17d52cf7c46b0a70da1b3172cec'


key_value,comprometido_2022,devengado_2022,girado_2022,comprometido_2023,devengado_2023,girado_2023,comprometido_2024,devengado_2024,girado_2024
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2a7fc17d52cf7c46b0a70da1b3172c…",0.0,0.0,0.0,0.0,0.0,0.0,120900.0,120900.0,120900.0
"""2a7fc17d52cf7c46b0a70da1b3172c…",17773.9,17773.9,17773.9,261850.0,260850.0,260850.0,0.0,0.0,0.0


🔄 Pivote de Arquitectura: Migración de Notebook a Script de Producción
El Desafío de Ingeniería:
Durante la fase de auditoría profunda, descubrimos que el dataset del MEF cuenta con 8,002,563 registros, de los cuales 7,842,582 son llaves únicas (key_value).

Intentar realizar una operación de agregación (group_by().agg()) con este nivel de cardinalidad y un ancho de tabla de 88 columnas directamente en la memoria RAM dentro de un entorno Jupyter Notebook provoca un colapso del Kernel por OOM (Out of Memory). Los notebooks mantienen estados en caché que fragmentan la RAM, haciéndolos inviables para transformaciones ETL pesadas de este calibre.

La Estrategia de Solución:
Decidimos desacoplar el pipeline de la capa Silver del notebook y migrarlo a un script de Python puro en la ruta ../src/transform_silver.py.

Para resolver la restricción de hardware, aplicamos una técnica de Particionamiento Determinista por Hash:

Dividimos el universo de datos en 16 sub-lotes independientes basados en el primer carácter hexadecimal del key_value (0-9, a-f).

Como los duplicados comparten el mismo hash, garantizamos que los clones caigan siempre en la misma partición.

Procesamos cada lote de manera secuencial liberando la memoria RAM con el recolector de basura (gc.collect()).

Finalmente, unificamos las particiones limpias en un único archivo Parquet optimizado.

In [ ]:
# Ejecutamos el script de producción desde el notebook para mantener la trazabilidad
%run ../src/transform_silver.py

📖 Leyendo diccionario de variables...
🚀 Iniciando procesamiento por partición de Hash (16 lotes)...
📦 [1/16] Procesando lote de llaves que inician con: '0'...
📦 [2/16] Procesando lote de llaves que inician con: '1'...
📦 [3/16] Procesando lote de llaves que inician con: '2'...
📦 [4/16] Procesando lote de llaves que inician con: '3'...
📦 [5/16] Procesando lote de llaves que inician con: '4'...
📦 [6/16] Procesando lote de llaves que inician con: '5'...
📦 [7/16] Procesando lote de llaves que inician con: '6'...
📦 [8/16] Procesando lote de llaves que inician con: '7'...
📦 [9/16] Procesando lote de llaves que inician con: '8'...
📦 [10/16] Procesando lote de llaves que inician con: '9'...
📦 [11/16] Procesando lote de llaves que inician con: 'a'...
📦 [12/16] Procesando lote de llaves que inician con: 'b'...
📦 [13/16] Procesando lote de llaves que inician con: 'c'...
📦 [14/16] Procesando lote de llaves que inician con: 'd'...
📦 [15/16] Procesando lote de llaves que inician con: 'e'...
📦 [16/16]

## 🔀 Paso 3 & 4: Transposición (Unpivot) y Creación de Clave Primaria Final

*Objetivo:* Transformar el dataset de su "Formato Ancho" (donde los años y métricas financieras son columnas independientes) a un "Formato Largo" adecuado para analítica e ingesta en un Modelo Estrella.

**Estrategia:**
1. Identificar dinámicamente las columnas numéricas que terminan en un año (ej. `_2022`, `_2023`, etc.).
2. Separar el nombre de la variable (ej. `pia`, `pim`, `devengado`) del año correspondiente (`anio`).
3. Aplicar la función `unpivot()` (antiguamente `melt`) de Polars para normalizar la tabla.
4. Generar el `sk_silver_id` definitivo sumando un Hash al nuevo nivel de granularidad por Año y Fase.

In [ ]:
import polars as pl
import re
import gc
from pathlib import Path

RUTA_SILVER_INTERMEDIO = Path("../data/02_silver/mef_consolidado_silver.parquet")
RUTA_SILVER_FINAL = Path("../data/02_silver/mef_final_silver.parquet")
TMP_DIR_UNPIVOT = Path("../data/00_tmp_unpivot")
TMP_DIR_UNPIVOT.mkdir(parents=True, exist_ok=True)

print("📖 Extrayendo esquema de la tabla...")
esquema = pl.read_parquet_schema(RUTA_SILVER_INTERMEDIO)
columnas_totales = list(esquema.keys())

columnas_dinero = [col for col in columnas_totales if re.search(r'_\d{4}$', col)]
columnas_descriptivas = [col for col in columnas_totales if col not in columnas_dinero]

print(f"💰 Se procesarán {len(columnas_dinero)} columnas financieras, ¡DE A POCOS (Una por una)!")

for i, col_dinero in enumerate(columnas_dinero, 1):
    print(f"📦 [{i}/{len(columnas_dinero)}] Procesando y filtrando: {col_dinero} ...")
    
    match = re.match(r'(.+)_(\d{4})$', col_dinero)
    fase = match.group(1)
    anio = int(match.group(2))
    
    lazy_chunk = (
        pl.scan_parquet(RUTA_SILVER_INTERMEDIO)
        .select(columnas_descriptivas + [col_dinero]) 
        .filter(pl.col(col_dinero) != 0)              
        .rename({col_dinero: "monto"})                
        .with_columns([
            pl.col("monto").cast(pl.Float64),         # <--- ¡LA SOLUCIÓN! Forzamos decimales siempre
            pl.lit(fase).alias("fase"),               
            pl.lit(anio).alias("anio").cast(pl.Int32) 
        ])
    )
    
    lazy_chunk = lazy_chunk.with_columns(
        (pl.col("key_value") + "_" + pl.col("anio").cast(pl.String) + "_" + pl.col("fase"))
        .hash()
        .alias("sk_silver_id")
    )
    
    cols_ordenadas = ["sk_silver_id"] + [c for c in lazy_chunk.collect_schema().names() if c != "sk_silver_id"]
    lazy_chunk = lazy_chunk.select(cols_ordenadas)
    
    archivo_salida = TMP_DIR_UNPIVOT / f"part_{col_dinero}.parquet"
    
    df_chunk = lazy_chunk.collect()
    df_chunk.write_parquet(archivo_salida, compression="zstd")
    
    del df_chunk
    gc.collect()

print("🔀 ¡Todas las columnas procesadas! Uniendo los bloques en el archivo final...")
# Juntamos los 35 pedacitos. Ahora Polars no se quejará porque TODOS los montos son Float64
lazy_unificado = pl.scan_parquet(TMP_DIR_UNPIVOT / "part_*.parquet")
lazy_unificado.sink_parquet(RUTA_SILVER_FINAL, compression="zstd")

print("✨ ¡CAPA SILVER COMPLETADA CON ÉXITO! Tu estrategia de ir 'de a pocos' triunfó. ✨")

📖 Extrayendo esquema de la tabla...
💰 Se procesarán 35 columnas financieras, ¡DE A POCOS (Una por una)!
📦 [1/35] Procesando y filtrando: pia_2022 ...
📦 [2/35] Procesando y filtrando: pim_2022 ...
📦 [3/35] Procesando y filtrando: certificado_2022 ...
📦 [4/35] Procesando y filtrando: comprometido_anual_2022 ...
📦 [5/35] Procesando y filtrando: comprometido_2022 ...
📦 [6/35] Procesando y filtrando: devengado_2022 ...
📦 [7/35] Procesando y filtrando: girado_2022 ...
📦 [8/35] Procesando y filtrando: pia_2023 ...
📦 [9/35] Procesando y filtrando: pim_2023 ...
📦 [10/35] Procesando y filtrando: certificado_2023 ...
📦 [11/35] Procesando y filtrando: comprometido_anual_2023 ...
📦 [12/35] Procesando y filtrando: comprometido_2023 ...
📦 [13/35] Procesando y filtrando: devengado_2023 ...
📦 [14/35] Procesando y filtrando: girado_2023 ...
📦 [15/35] Procesando y filtrando: pia_2024 ...
📦 [16/35] Procesando y filtrando: pim_2024 ...
📦 [17/35] Procesando y filtrando: certificado_2024 ...
📦 [18/35] Proces